In [1]:
import sys
sys.path.append('..')
import os
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import linregress
import cartopy
import cartopy.crs as ccrs
import matplotlib.patches as patches
from src.inversion_scripts.utils import plot_field, sum_total_emissions, get_posterior_emissions, get_mean_emissions

In [2]:
state_vector_filepath = "/n/holylfs06/LABS/jacob_lab2/Lab/mhe/Global_2022_annual/StateVector.nc"

state_vector = xr.open_dataset(state_vector_filepath)
state_vector_labels = state_vector["StateVector"]

# Identify the last element of the region of interest
last_ROI_element = int(
    np.nanmax(state_vector_labels.values)
)

# Define mask for region of interest
mask = state_vector_labels <= last_ROI_element

In [ ]:
# Get posterior emissions for each ensemble member and put into a single xarray dataset

def compute_posterior_ensemble(prior_ds, ens_scale_ds):
    ensembles = ens_scale_ds.sizes["ensemble"]
    posterior_ensemble_list = []

    for imem in range(ensembles):
        scale_i = ens_scale_ds["ScaleFactor"].isel(ensemble=imem)
        posterior_i = get_posterior_emissions(prior_ds, scale_i)

        # Add the ensemble index as a coordinate for stacking later
        posterior_i = posterior_i.expand_dims(ensemble=[imem])
        posterior_ensemble_list.append(posterior_i)

    # Combine all posterior datasets along the ensemble dimension
    posterior_ds = xr.concat(posterior_ensemble_list, dim="ensemble")
    return posterior_ds


In [22]:
years = [2019, 2020, 2021, 2022, 2023, 2024]
# years = [2019]

for year in years:
    total_emissions_list = []

    start_date = f'{year}0101'
    end_date = f'{year+1}0101'

    if year == 2019:
        prior_cache = f"/n/holylfs06/LABS/jacob_lab2/Lab/mhe/Global_{year}_annual_ResMefix_2/hemco_prior_emis/OutputDir"
        ens_scale_ds_path = f'/n/holylfs06/LABS/jacob_lab2/Lab/mhe/Global_{year}_annual_ResMefix_2/inversion/gridded_posterior_ensemble.nc'
    else:
        prior_cache = f"/n/holylfs06/LABS/jacob_lab2/Lab/mhe/Global_{year}_annual/hemco_prior_emis/OutputDir"
        ens_scale_ds_path = f'/n/holylfs06/LABS/jacob_lab2/Lab/mhe/Global_{year}_annual/inversion/gridded_posterior_ensemble.nc'


    prior_ds = get_mean_emissions(start_date, end_date, prior_cache)
    areas = prior_ds["AREA"]
    ens_scale_ds = xr.open_dataset(ens_scale_ds_path)

    # calculate posterior emissions for each ensemble member
    posterior_ds = compute_posterior_ensemble(prior_ds, ens_scale_ds)

    # save posterior_ds to posterior_ds_ensembles folder
    output_dir = 'posterior_ds_ensembles'
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, f'posterior_ds_ensemble_{year}.nc')
    posterior_ds.to_netcdf(output_path)
    print(f'Saved {year}')

Saved 2019
Saved 2020
Saved 2021
Saved 2022
Saved 2023
Saved 2024


In [3]:
def ensemble_trend_range(mask, sector):
    '''Returns the lower bound and upper bound of the linear regression trend from the ensemble for a given region'''
    years = [2019, 2020, 2021, 2022, 2023, 2024]
    n_ensembles = 27
    ensemble_emissions_by_member = [[] for _ in range(n_ensembles)]
    for year in years:
        # set directory
        path = f'posterior_ds_ensembles/posterior_ds_ensemble_{year}.nc'
        posterior_ds = xr.load_dataset(path)
        for imem in range(n_ensembles):
            posterior_i = posterior_ds.sel(ensemble=imem)
            areas = posterior_i["AREA"]
            if sector == 'Fuel/Waste':
                fuel_waste = posterior_i["EmisCH4_Oil"] + posterior_i["EmisCH4_Gas"] + posterior_i["EmisCH4_Wastewater"] + posterior_i["EmisCH4_Landfills"]
                emis_i = sum_total_emissions(fuel_waste, areas, mask) # Tg/yr
            else:
                emis_i = sum_total_emissions(posterior_i[f"EmisCH4_{sector}"], areas, mask) # Tg/yr
            ensemble_emissions_by_member[imem].append(emis_i)

    slopes = []
    for emissions in ensemble_emissions_by_member:
        result = linregress(years, emissions)
        slopes.append(result.slope)

    min_slope = min(slopes)
    max_slope = max(slopes)

    print(f"Min slope: {min_slope:.2f} Tg/yr")
    print(f"Max slope: {max_slope:.2f} Tg/yr")

In [53]:
# East Africa
mask_eastafrica = xr.open_dataset("../shapefiles/masks/east_africa.nc")
mask_eastafrica = mask_eastafrica.rename({"__xarray_dataarray_variable__": "Mask"})
mask_eastafrica = mask_eastafrica["Mask"]

print(ensemble_trend_range(mask_eastafrica, 'Total'))
print(ensemble_trend_range(mask_eastafrica, 'Livestock'))

Min slope: 1.40 Tg/yr
Max slope: 1.65 Tg/yr
None
Min slope: 0.53 Tg/yr
Max slope: 0.75 Tg/yr
None


In [7]:
# South America
mask_SA = xr.open_dataset("../shapefiles/masks/SA.nc")
mask_SA = mask_SA.rename({"__xarray_dataarray_variable__": "Mask"})
mask_SA = mask_SA["Mask"]

print(ensemble_trend_range(mask_SA, 'Total'))
print(ensemble_trend_range(mask_SA, 'Fuel/Waste'))

Min slope: -0.36 Tg/yr
Max slope: 2.61 Tg/yr
None
Min slope: 0.58 Tg/yr
Max slope: 0.80 Tg/yr
None


In [50]:
# Europe
mask_europe = xr.open_dataset("../shapefiles/masks/europe.nc")
mask_europe = mask_europe.rename({"__xarray_dataarray_variable__": "Mask"})
mask_europe = mask_europe["Mask"]

ensemble_trend_range(mask_europe, 'Total')

Min slope: 0.87 Tg/yr
Max slope: 1.45 Tg/yr


In [5]:
# Africa
mask_africa = xr.open_dataset("../shapefiles/masks/africa.nc")
mask_africa = mask_africa.rename({"__xarray_dataarray_variable__": "Mask"})
mask_africa = mask_africa["Mask"]

ensemble_trend_range(mask_africa, 'Total')

Min slope: 0.76 Tg/yr
Max slope: 1.33 Tg/yr


In [5]:
# Sudd
lats = [5, 10]
lons = [28, 34.5]

years = [2019, 2020, 2021, 2022, 2023, 2024]
n_ensembles = 27
ensemble_emissions_by_member = [[] for _ in range(n_ensembles)]
for year in years:
    # set directory
    path = f'posterior_ds_ensembles/posterior_ds_ensemble_{year}.nc'
    posterior_ds = xr.load_dataset(path)
    for imem in range(n_ensembles):
        posterior_i = posterior_ds.sel(ensemble=imem)
        areas = posterior_i["AREA"]
        # select grid cells only within lat and lon bounds in posterior
        posterior_subset = posterior_i.where((posterior_i.lat > lats[0]) 
                                    & (posterior_i.lat < lats[1]) 
                                    & (posterior_i.lon > lons[0]) 
                                    & (posterior_i.lon < lons[1]), drop=True)
        # select grid cells only in lat and lon bounds in areas
        areas_subset = areas.where((areas.lat > lats[0])
                        & (areas.lat < lats[1]) 
                        & (areas.lon > lons[0]) 
                        & (areas.lon < lons[1]), drop=True)
        emis_i = sum_total_emissions(posterior_subset[f"EmisCH4_Total"], areas_subset, mask) # Tg/yr
        ensemble_emissions_by_member[imem].append(emis_i)

slopes = []
for emissions in ensemble_emissions_by_member:
    result = linregress(years, emissions)
    slopes.append(result.slope)

min_slope = min(slopes)
max_slope = max(slopes)

print(f"Min slope: {min_slope:.2f} Tg/yr")
print(f"Max slope: {max_slope:.2f} Tg/yr")   

Min slope: 0.18 Tg/yr
Max slope: 0.27 Tg/yr


In [51]:
#CONUS
mask_conus = xr.open_dataset("../shapefiles/masks/conus.nc")
mask_conus = mask_conus.rename({"__xarray_dataarray_variable__": "Mask"})
mask_conus = mask_conus["Mask"]

ensemble_trend_range(mask_conus, 'Total')

Min slope: -0.59 Tg/yr
Max slope: 0.39 Tg/yr


In [48]:
# Colombia
mask_colombia = xr.open_dataset("../shapefiles/masks/colombia.nc")
mask_colombia = mask_colombia.rename({"__xarray_dataarray_variable__": "Mask"})
mask_colombia = mask_colombia["Mask"]

ensemble_trend_range(mask_colombia, 'Total')

Min slope: 0.16 Tg/yr
Max slope: 0.40 Tg/yr


In [49]:
# Brazil
mask_brazil = xr.open_dataset("../shapefiles/masks/brazil.nc")
mask_brazil = mask_brazil.rename({"__xarray_dataarray_variable__": "Mask"})
mask_brazil = mask_brazil["Mask"]

ensemble_trend_range(mask_brazil, 'Total')

Min slope: -0.76 Tg/yr
Max slope: 0.11 Tg/yr


In [44]:
# Amazon basin
mask_amazon = xr.open_dataset("../shapefiles/masks/amazon.nc")
mask_amazon = mask_amazon.rename({"__xarray_dataarray_variable__": "Mask"})
mask_amazon = mask_amazon["Mask"]

ensemble_trend_range(mask_amazon, 'Wetlands')

Min slope: -3.52 Tg/yr
Max slope: -2.12 Tg/yr


In [52]:
#China
mask_china = xr.open_dataset("../shapefiles/masks/china.nc")
mask_china = mask_china.rename({"__xarray_dataarray_variable__": "Mask"})
mask_china = mask_china["Mask"]

ensemble_trend_range(mask_china, 'Total')

Min slope: 0.13 Tg/yr
Max slope: 0.59 Tg/yr


In [45]:
# Hudson Bay Lowlands
mask_hbl = xr.open_dataset("../shapefiles/masks/hbl.nc")
mask_hbl = mask_hbl.rename({"__xarray_dataarray_variable__": "Mask"})
mask_hbl = mask_hbl["Mask"]

ensemble_trend_range(mask_hbl, 'Wetlands')

Min slope: -0.24 Tg/yr
Max slope: -0.05 Tg/yr


In [47]:
# West Siberian Lowlands
mask_wsl = xr.open_dataset("../shapefiles/masks/wsl.nc")
mask_wsl = mask_wsl.rename({"__xarray_dataarray_variable__": "Mask"})
mask_wsl = mask_wsl["Mask"]

ensemble_trend_range(mask_wsl, 'Wetlands')

Min slope: -0.48 Tg/yr
Max slope: -0.30 Tg/yr


In [4]:
# Russia
mask_russia = xr.open_dataset("../shapefiles/masks/russia.nc")
mask_russia = mask_russia.rename({"__xarray_dataarray_variable__": "Mask"})
mask_russia = mask_russia["Mask"]

ensemble_trend_range(mask_russia, 'Total')

Min slope: -1.00 Tg/yr
Max slope: -0.71 Tg/yr


In [5]:
# Middle East
mask_middle_east = xr.open_dataset("../shapefiles/masks/middle-east.nc")
mask_middle_east = mask_middle_east.rename({"__xarray_dataarray_variable__": "Mask"})
mask_middle_east = mask_middle_east["Mask"]

ensemble_trend_range(mask_middle_east, 'Total')

Min slope: 0.29 Tg/yr
Max slope: 0.53 Tg/yr


In [10]:
# South+Southeast Asia
mask_south_asia = xr.open_dataset("../shapefiles/masks/south-asia.nc")
mask_south_asia = mask_south_asia.rename({"__xarray_dataarray_variable__": "Mask"})
mask_south_asia = mask_south_asia["Mask"]

print(ensemble_trend_range(mask_south_asia, 'Total'))

mask_southeast_asia = xr.open_dataset("../shapefiles/masks/southeast-asia.nc")
mask_southeast_asia = mask_southeast_asia.rename({"__xarray_dataarray_variable__": "Mask"})
mask_southeast_asia = mask_southeast_asia["Mask"]

print(ensemble_trend_range(mask_southeast_asia, 'Total'))


Min slope: -1.65 Tg/yr
Max slope: 0.16 Tg/yr
None
Min slope: 0.08 Tg/yr
Max slope: 0.97 Tg/yr
None
